# CA-TNN demo

This notebook restores one Pavia Centre cube with 95% of the entries missing. Run the cells from top to bottom, with the working directory set to the repository root. A CUDA GPU is required: tensors are created as `torch.cuda.FloatTensor`.

Training writes `CA-TNN_Pavia_MR95_<YYYY-MM-DD HH.MM>/` next to the notebook. That folder contains the checkpoint with the highest mean per-band PSNR, the restored cube, a CSV of the run, and the loss, PSNR, and nuclear-norm curves.


In [ ]:
import os
import torch
from torch import nn, optim 
from torch.autograd import Variable 
from skimage.metrics import peak_signal_noise_ratio
from torch.nn import init
get_ipython().run_line_magic('matplotlib', 'inline')
import numpy as np 
import scipy.io as scio
import matplotlib.pyplot as plt 
import scipy.io
import math
import time
import pandas as pd
import datetime
import copy

## Band-wise PSNR

`psnr3d` averages `peak_signal_noise_ratio` over the spectral bands of a cube shaped `(height, width, bands)`. The complete cube is used for this score and for keeping the best checkpoint. The loss in the training cell is computed from the observed cube and the mask.


In [ ]:
def psnr3d(x,y): 
    ps = 0
    for i in range(x.shape[2]):
        ps += peak_signal_noise_ratio(x[:,:,i], y[:,:,i])
    return ps/x.shape[2]

dtype = torch.cuda.FloatTensor


## Data and transform size

`dataset/Pavia-MR95/` stores three `float64` arrays of shape `(200, 200, 80)` (height, width, bands). Values of the complete cube lie in `[0, 1]`.

| File | Contents |
| --- | --- |
| `Pavia_gtruth.npy` | Complete cube |
| `Pavia_mask.npy` | `1` on observed entries and `0` on missing entries. About 5% of the entries are observed |
| `Pavia_sampled.npy` | Observed cube: pixels where the mask is `1`, and `0` elsewhere |

`X` is the observed cube on GPU, `X_GTruth` is the complete cube on GPU, and `mask` is a CUDA float tensor.

`band_shrink` sets the latent band count `n_4 = n_3 * band_shrink`. With `band_shrink = 1` the latent cube has 80 bands. `CNN_layer` is the number of depthwise spatial convolutions, and `CNN_size` is their kernel width.

Another cube can be loaded by pointing `folder` at a directory with `<data_set>_gtruth.npy`, `<data_set>_sampled.npy`, and `<data_set>_mask.npy`, then setting `data_set` and `case`. The three arrays need the same shape.


In [ ]:
# load npy files
data_set = 'Pavia'
case = 'MR95'
folder = 'dataset/Pavia-MR95/'
img = np.load(folder + data_set + '_''gtruth.npy')
sampled_image = np.load(folder + data_set + '_''sampled.npy')
mask = np.load(folder + data_set + '_''mask.npy')

print('data set: ' + data_set)
print('case: ' + case)

#set data
X_np = sampled_image 
X = torch.from_numpy(X_np).type(dtype).cuda()

mask = torch.from_numpy(mask ).type(dtype)

X_GTruth_np=img 
X_GTruth = torch.from_numpy(X_GTruth_np).type(dtype).cuda()

dtype = torch.cuda.FloatTensor

# set network parameters
band_shrink = 1
CNN_layer = 1
CNN_size = 3

# set some parameters
label = 'CA-TNN_' + data_set + '_' + case
path0 = ""

n_1 = X.shape[0]
n_2 = X.shape[1]
n_3 = X.shape[2]

n_4 = n_3*band_shrink

## Model

Training uses `Y_net_CoNot_mode3`.

- `X_hat` is the latent cube, an `nn.Parameter` of shape `(n_1, n_2, n_4)`, drawn from `Uniform(-1/sqrt(n_1), 1/sqrt(n_1))`.
- `SEatt_av_fc1` average-pools each frontal slice, then applies a bias-free linear layer and a sigmoid.
- `expFC` stores the logarithm of its weight and bias and applies `exp` in `forward`, so the channel weights stay positive.
- `SEatt_w` computes those weights from a detached copy of the latent cube, so this path updates the excitation parameters only.
- `my_expSEAttention` multiplies each frontal slice by its weight. The hidden width equals `n_4` because `reduction = 1`.
- `CNN_FC_block` maps the latent cube to the image. The spatial stage is a depthwise `Conv2d` (`groups = n_4`, no bias, padding that keeps the spatial size) followed by `LeakyReLU`. The spectral stage is a bias-free `Linear` from `n_4` to `n_3`, again followed by `LeakyReLU`.

`forward` returns the weighted latent cube and the reconstruction. The reconstruction is `g(X_hat)`, taken from the latent cube before the channel weights are applied. `F_norm` is an `MSELoss` and is the data term. `SSNT` is defined in this cell and is not called.


In [ ]:
#define networks
#define expSEatt
class SEatt_av_fc1(nn.Module):
    '''
    weighting at the 3rd mode
    '''
    def __init__(self, channel,reduction):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=False),
            nn.Sigmoid()
        )

        self.init_weights()

    def init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                init.kaiming_uniform_(
                    m.weight)
                if m.bias is not None:
                    init.constant_(m.bias, 0)

    def forward(self, x):
        b, _, _ = x.size()
        y = self.avg_pool(x).view(1,b)
        y = self.fc1(y).view(b, 1)
        return y
    
class expFC(nn.Module):
    def __init__(self, in_channel,out_channel,bias: bool = True, device=None, dtype=None):
        factory_kwargs = {'device': device, 'dtype': dtype}
        super(expFC, self).__init__()
        self.weight = nn.Parameter(torch.empty((out_channel, in_channel), **factory_kwargs))
        if bias:
            self.bias = nn.Parameter(torch.empty(out_channel, **factory_kwargs))
        else:
            self.register_parameter('bias', None)
        self.reset_parameters()
    
    def reset_parameters(self) -> None:
        # Setting a=sqrt(5) in kaiming_uniform is the same as initializing with
        # uniform(-1/sqrt(in_features), 1/sqrt(in_features)). For details, see
        # https://github.com/pytorch/pytorch/issues/57109
        init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        weight0=self.weight-self.weight.data.min()+1e-8
        self.weight=nn.Parameter(torch.log(weight0))
        
        if self.bias is not None:
            fan_in, _ = init._calculate_fan_in_and_fan_out(self.weight)
            bound = 1 / math.sqrt(fan_in) if fan_in > 0 else 0
            init.uniform_(self.bias, a=-bound,b= bound)

    def forward(self, x):
        y=nn.functional.linear(x,torch.exp(self.weight),bias=torch.exp(self.bias))
        return y
    
class SEatt_w(nn.Module): 
    # input: the whole tensor (channel mode as index 2)
    # output: weights on different frontal slices
    def __init__(self,in_channel,reduction=1):
        super(SEatt_w, self).__init__()
        self.av_fc1 = SEatt_av_fc1(channel=in_channel, reduction=reduction)
                                    
        self.expfc = expFC(in_channel // reduction,in_channel,bias=True)
                                    
    def forward(self, x):
        x = x.detach().clone()
        x = x.permute(2,0,1)
        y1=self.av_fc1(x).permute(1,0)
        y2=self.expfc(y1)
        return y2

class my_expSEAttention(nn.Module): 
    # input: the whloe tensor
    # output: weighted tensor with weights on frontal slices
    def __init__(self,in_channel,reduction):
        super(my_expSEAttention, self).__init__()
        self.SEatt_w = SEatt_w(in_channel,reduction)
                                    
    def forward(self, x):
        y=self.SEatt_w(x)
        x=x.permute(0,2,1)
        out=x * (y.permute(1,0).expand_as(x))
        return out.permute(0,2,1)

class CNN_FC_block(nn.Module):
    def __init__(self,n_4,n_0,CNN_layer,CNN_size):
        super(CNN_FC_block, self).__init__()
        self.cnn_block=nn.Sequential(nn.Conv2d(n_4,n_4,CNN_size,padding=int((CNN_size-1)/2),groups=n_4,bias=False),
                               nn.LeakyReLU())   
        for l in range(CNN_layer-1):
            self.cnn_block=nn.Sequential(self.cnn_block,
                                         nn.Conv2d(n_4,n_4,CNN_size,padding=int((CNN_size-1)/2),groups=n_4,bias=False),
                                         nn.LeakyReLU())      
           
        self.fc_block=nn.Sequential(nn.Linear(n_4,n_0, bias = False),
                              nn.LeakyReLU())
    def forward(self, x):
        special_g=self.cnn_block(x.permute(2,0,1))
        g_x=self.fc_block(special_g.permute(1,2,0))
        return g_x    

class Y_net_CoNot_mode3(nn.Module): 
    def __init__(self,n_1,n_2,n_3,n_4,CNN_layer,CNN_size):
        # n_4: dimension of the latent space
        super(Y_net_CoNot_mode3, self).__init__()
        self.X_hat = nn.Parameter(torch.Tensor(n_1,n_2,n_4))
        self.atten=my_expSEAttention(in_channel=n_4,reduction=1)
        self.g = CNN_FC_block(n_4, n_3,CNN_layer,CNN_size)
        
        self.reset_parameters()

    def reset_parameters(self):
        stdv = 1. / math.sqrt(self.X_hat.size(0))
        self.X_hat.data.uniform_(-stdv, stdv)
                                    
    def forward(self):
        X_hat=self.X_hat
        weighted_X_hat=self.atten(X_hat)
        g_x_uprgt=self.g(self.X_hat)
        return weighted_X_hat, g_x_uprgt

class SSNT(nn.Module): 
    def __init__(self,n_4,n_3):
        super(SSNT, self).__init__()
        self.f = nn.Sequential(  NoFC_3(n_3,n_4),
                                NoFC_3(n_4,n_4))
                                    
        self.g = nn.Sequential(NoFC_3(n_4,n_4),
                               NoFC_3(n_4,n_3))
                                    
    def forward(self, x):
        f_x = self.f(x)
        g_x = self.g(f_x)
        return f_x, g_x
    
F_norm = nn.MSELoss()


## Saved files

`save_training_result` creates the output folder and writes:

- a CSV with the learning rate, `beta1`, `lambda`, `band_shrink`, `CNN_layer`, `CNN_size`, the last iteration index, wall time, PSNR at the last iterate (`PSNR_stop`), and the best PSNR (`PSNR_max`)
- `Training Loss1.png` and `Training PSNR1.png` for the whole run, and `Training Loss2.png` and `Training PSNR2.png` for the last four fifths of iterations
- `nuc_.png` for the nuclear-norm term. The second `savefig` uses the same name, so the file shows the last four fifths of the curve
- `<label>.pth`, the `state_dict` of the iterate with the highest PSNR
- `<data_set>_<case>_CNN_FC_expSE_slice.npy`, the restored cube at that iterate


In [ ]:
def save_training_result(model_state, X_save, df, nucs,\
            losses, psnr_curve, iter):
    now_time = datetime.datetime.now().strftime('%Y-%m-%d %H.%M')
    path=label+"_"+now_time
    try:
        os.makedirs(path)
    except:
        now_time = datetime.datetime.now().strftime('%m-%d %H.%M.%S')
        path=label+"_"+now_time
        os.makedirs(path)
        
    path_save=path0+path+"/"
    df.to_csv(path_save + label+"_"+now_time+".csv",index=False)


    start=int(iter/5)

    plt.plot(losses)
    plt.xlabel('Iteration')
    plt.ylabel('Loss')
    plt.title('Training Loss')
    plt.savefig(path_save+"Training Loss1")
    plt.show()

    plt.plot(start+np.arange(np.array(losses).shape[0]-start),losses[start:])
    plt.xlabel('Iteration')
    plt.ylabel('Loss')
    plt.title('Training Loss')
    plt.savefig(path_save+"Training Loss2")
    plt.show()

    plt.plot(psnr_curve)
    plt.xlabel('Iteration')
    plt.ylabel('PSNR')
    plt.title('Training PSNR')
    plt.savefig(path_save+"Training PSNR1")
    plt.show()

    plt.plot(start+np.arange(np.array(psnr_curve).shape[0]-start),psnr_curve[start:])
    plt.xlabel('Iteration')
    plt.ylabel('PSNR')
    plt.title('Training PSNR')
    plt.savefig(path_save+"Training PSNR2")
    plt.show()

    plt.plot(nucs)
    plt.xlabel('Iteration')
    plt.title('nuc')
    plt.savefig(path_save+"nuc_")
    plt.show()

    plt.plot(start+np.arange(np.array(nucs).shape[0]-start),nucs[start:])
    plt.xlabel('Iteration')
    plt.title('nuc')
    plt.savefig(path_save+"nuc_")
    plt.show()

    torch.save({'model':model_state},path_save + label + '.pth')
    np.save(path_save + data_set + '_' + case + '_CNN_FC_expSE_slice.npy',X_save)

## One training iteration

`one_training` builds `Y_net_CoNot_mode3` and steps Adam on every parameter. `beta2` is `0.999`.

1. Reconstruct the image and compute the mean per-band PSNR. When the score improves, the image and the `state_dict` are kept for the files saved at the end.
2. Penalize the nuclear norm of one frontal slice of the weighted latent cube. The slice index is `iter % n_4`, so the bands are visited in turn. If that norm fails, the next slice is used instead.
3. Add the data term `F_norm(X_Out * mask, X * mask)`, the MSE on observed entries. The loss is `lambda_ * nuc + f`.
4. Take one Adam step.

Every 40 iterations the cell prints the iteration, the time spent on those 40 steps, and the current PSNR.


In [ ]:
# model initialization 
def init_model(Y_net_CoNot_mode3):

    model = Y_net_CoNot_mode3(n_1,n_2,n_3,n_4,CNN_layer,CNN_size).type(dtype)

    params = []
    params += [x for x in model.parameters()] 

    s = sum([np.prod(list(p.size())) for p in params]); 
    print('Number of params: %d' % s)

    return model, params

def one_training(max_iter, \
    lr, lambda_,  beta1):  
    #training

    model, params = init_model(Y_net_CoNot_mode3)
    optimizier = optim.Adam(params, lr=lr, betas=(beta1,0.999)) 

    psnr_curve = []
    losses = []
    fdltys = []
    nucs = []
    psnr_best = 0

    start_time=time.time()
    start_time_1 = time.time()
    for iter in range(max_iter):
        
        weighted_X_hat, X_Out = model()
        psnr = psnr3d(X_GTruth.cpu().detach().numpy(),X_Out.cpu().detach().numpy())
        
        if psnr > psnr_best:
            psnr_best = psnr
            X_save = X_Out.clone().cpu().detach().numpy()
            model_to_save = copy.deepcopy(model).cpu()
            model_state = model_to_save.state_dict() 
            
        nuc= 0
        try:    
            nuc += torch.norm(weighted_X_hat[:,:,int(iter%n_4)].cuda(),'nuc')
        except:
            nuc += torch.norm(weighted_X_hat[:,:,int((iter+1)%n_4)].cuda(),'nuc')
        #optimize the nuclear norm of only one frontal slice at one time
        
        f = F_norm(X_Out*mask,X*mask)

        loss = lambda_*nuc + f 

        optimizier.zero_grad()
        loss.backward(retain_graph=True)
        optimizier.step()

        psnr_curve.append(psnr.item())
        losses.append(loss.item())
        fdltys.append(f.item())
        nucs.append(nuc.item())
        

        if iter % 40 == 0:
            end_time=time.time()
            iteration_time=end_time-start_time
            start_time=time.time()
            print('training: ' + label)
            print('iter: {:}/{:}; time used: {:.3f}; PSNR: {:.3f}'.format(iter, max_iter, iteration_time, psnr))
            print()
            
            
    end_time_1 = time.time()
    training_time = end_time_1-start_time_1
    training_time = round(training_time,2)
    
    weighted_X_hat, X_Out = model()
    psnr = psnr3d(X_GTruth.cpu().detach().numpy(),X_Out.cpu().detach().numpy())
    psnr_max = max(psnr_curve)

    df = pd.DataFrame(list(zip([lr],[beta1], [lambda_], \
    [band_shrink],[CNN_layer],[CNN_size],\
    [iter], [training_time],[psnr],[psnr_max])),\
    columns =['learning rate','beta1','lambda',\
    'band_shrink','CNN_layer',\
    'CNN_size', 'iter','time_used', 'PSNR_stop','PSNR_max'])

    save_training_result(model_state, X_save, df, nucs,\
            losses, psnr_curve, iter)
            
    return model

## Run configuration

The cell below sets the optimizer arguments.

| Name | Value used | Role |
| --- | --- | --- |
| `max_iter` | `10000` | Number of Adam steps. The earlier assignment of `35000` is replaced by this one |
| `lr` | `1e-2` | Adam learning rate |
| `lambda_` | `1e-4` | Coefficient of the nuclear-norm term |
| `beta1` | `0.9` | Adam coefficient on the first moment |

The commented assignment `beta1 = 0.99` and `max_iter = 35000` is a longer run. In the paper, `λ` is chosen by the best PSNR: `1e-3` on Pavia Centre and `1e-6` on Washington DC Mall.


In [ ]:
#training
max_iter = 35000
lr = math.pow(10,-2)
lambda_ = math.pow(10, -4)
beta1 = 0.9
max_iter = 10000

# more accurate but needs more time
# beta1 = 0.99
# max_iter = 35000

model = one_training(max_iter, \
    lr, lambda_,  beta1)